# Double ML causal inference

### Импорты

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns

import doubleml as dml

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict

import xgboost as xgb
import optuna

from tqdm.auto import tqdm

alpha = 0.1
sns.set_style("whitegrid")
optuna.logging.set_verbosity(optuna.logging.WARNING)

### Данные

In [ ]:
df = pd.read_csv("data/final_dataset.csv")
df_men = df[df["sex"] == 1]
df_women = df[df["sex"] == 2]

In [ ]:
df_men

In [ ]:
df_women

### Вспомогательные функции и сетки гиперпараметров

In [ ]:
def rf_space(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 20),
    }


def xgb_space(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
    }


optuna_settings = {
    "n_trials": 20,
    "show_progress_bar": False,
}


def fit_irm(dml_data, model, score):
    if model == "LDA":
        ml_g = LinearDiscriminantAnalysis()
        ml_m = LinearDiscriminantAnalysis()
    elif model == "RF":
        ml_g = RandomForestClassifier(random_state=42, n_jobs=-1)
        ml_m = RandomForestClassifier(random_state=42, n_jobs=-1)
    elif model == "XGB":
        ml_g = xgb.XGBClassifier(
            random_state=42, verbosity=0, eval_metric="logloss", device="cuda"
        )
        ml_m = xgb.XGBClassifier(
            random_state=42, verbosity=0, eval_metric="logloss", device="cuda"
        )

    irm = dml.DoubleMLIRM(
        dml_data,
        ml_g=ml_g,
        ml_m=ml_m,
        trimming_threshold=0.05,
        n_folds=5,
        score=score,
    )

    if model == "RF":
        irm.tune_ml_models(
            ml_param_space={"ml_g": rf_space, "ml_m": rf_space},
            optuna_settings=optuna_settings,
        )
    elif model == "XGB":
        irm.tune_ml_models(
            ml_param_space={"ml_g": xgb_space, "ml_m": xgb_space},
            optuna_settings=optuna_settings,
        )

    irm.fit(n_jobs_cv=-1)
    return irm


def print_all_models(dml_data, label=""):
    for model in ["LDA", "RF", "XGB"]:
        for score in ["ATE", "ATTE"]:
            irm = fit_irm(dml_data, model, score)
            header = f"{label} | {model} | {score}".strip(" |")
            print(f"--- {header} ---")
            print(irm.summary)
            print(irm.confint(level=1 - alpha))
            print()

### Женщины и гипертония

In [ ]:
treatment = "diploma"
outcome = "hypertension"
controls = [
    "age",
    "mar_st",
    "invalid",
    "type_area",
    "income",
    "n_child",
    "alcohol",
    "smoking",
    "phys_active",
]

In [ ]:
dml_data_women = dml.DoubleMLData(
    df_women, y_col=outcome, d_cols=treatment, x_cols=controls
)
dml_data_men = dml.DoubleMLData(
    df_men, y_col=outcome, d_cols=treatment, x_cols=controls
)

In [ ]:
print_all_models(dml_data_women, label="Women + hypertension")

### Женщины и заболевания глаз

In [ ]:
treatment = "diploma"
outcome = "eyes"
controls = [
    "age",
    "mar_st",
    "invalid",
    "type_area",
    "income",
    "n_child",
    "alcohol",
    "smoking",
    "phys_active",
]

In [ ]:
dml_data_women = dml.DoubleMLData(
    df_women, y_col=outcome, d_cols=treatment, x_cols=controls
)
dml_data_men = dml.DoubleMLData(
    df_men, y_col=outcome, d_cols=treatment, x_cols=controls
)

In [ ]:
print_all_models(dml_data_women, label="Women + eyes")

### Женщины и аллергия

In [ ]:
treatment = "diploma"
outcome = "allergy"
controls = [
    "age",
    "mar_st",
    "invalid",
    "type_area",
    "income",
    "n_child",
    "alcohol",
    "smoking",
    "phys_active",
]

In [ ]:
dml_data_women = dml.DoubleMLData(
    df_women, y_col=outcome, d_cols=treatment, x_cols=controls
)
dml_data_men = dml.DoubleMLData(
    df_men, y_col=outcome, d_cols=treatment, x_cols=controls
)

In [ ]:
print_all_models(dml_data_women, label="Women + allergy")

### Мужчины и оценка состояния здоровья

In [ ]:
treatment = "diploma"
outcome = "is_health_very_good"
controls = [
    "age",
    "mar_st",
    "invalid",
    "type_area",
    "income",
    "n_child",
    "alcohol",
    "smoking",
    "phys_active",
]

In [ ]:
dml_data_women = dml.DoubleMLData(
    df_women, y_col=outcome, d_cols=treatment, x_cols=controls
)
dml_data_men = dml.DoubleMLData(
    df_men, y_col=outcome, d_cols=treatment, x_cols=controls
)

In [ ]:
print_all_models(dml_data_men, label="Men + is_health_very_good")

### Другие интересные наблюдения

In [ ]:
treatment = "diploma"
controls = [
    "age",
    "mar_st",
    "invalid",
    "type_area",
    "income",
    "n_child",
    "alcohol",
    "smoking",
    "phys_active",
]

diseases = [
    "heart",
    "lungs",
    "liver",
    "kidneys",
    "stomach",
    "spine",
    "diabetes",
    "hypertension",
    "joints",
    "ENT_organs",
    "neurology",
    "eyes",
    "allergy",
    "veins",
    "skin",
    "oncology",
    "is_health_good",
    "is_health_very_good",
]

results = []

for outcome in tqdm(diseases):
    dml_data_w = dml.DoubleMLData(
        df_women, y_col=outcome, d_cols=treatment, x_cols=controls
    )
    dml_data_m = dml.DoubleMLData(
        df_men, y_col=outcome, d_cols=treatment, x_cols=controls
    )

    for gender, dml_data in [("Women", dml_data_w), ("Men", dml_data_m)]:
        for model in ["LDA", "RF", "XGB"]:
            for score in ["ATE", "ATTE"]:
                irm = fit_irm(dml_data, model, score)
                p_val = irm.pval.item()
                ci = irm.confint(level=1 - alpha)
                results.append({
                    "outcome": outcome,
                    "gender": gender,
                    "model": model,
                    "score": score,
                    "coef": irm.coef.item(),
                    "se": irm.se.item(),
                    "p_val": p_val,
                    "ci_low": ci.iloc[0, 0],
                    "ci_high": ci.iloc[0, 1],
                    "significant": p_val <= alpha,
                })

results_df = pd.DataFrame(results)
print(
    f"Collected {len(results_df)} estimates "
    f"({len(diseases)} outcomes x 2 genders x 3 models x 2 scores)."
)

In [ ]:
sig = results_df[results_df["significant"]].copy()
print(f"Total significant estimates (p <= {alpha}): {len(sig)}\n")

for (gender, outcome), grp in sig.groupby(["gender", "outcome"], sort=False):
    print(f"{gender} - {outcome}:")
    for _, r in grp.iterrows():
        print(
            f"  {r['model']:3s} | {r['score']:4s} | "
            f"coef={r['coef']:+.4f}  p={r['p_val']:.4f}  "
            f"CI=[{r['ci_low']:+.4f}, {r['ci_high']:+.4f}]"
        )
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 14), sharey=True)

for ax, sc in zip(axes, ["ATE", "ATTE"]):
    sub = results_df[
        (results_df["model"] == "RF") & (results_df["score"] == sc)
    ].reset_index(drop=True)

    for i, r in sub.iterrows():
        if r["significant"]:
            color = "#e74c3c"
        elif r["gender"] == "Women":
            color = "#3498db"
        else:
            color = "#2ecc71"

        ax.errorbar(
            r["coef"],
            i,
            xerr=[[r["coef"] - r["ci_low"]], [r["ci_high"] - r["coef"]]],
            fmt="o",
            color=color,
            capsize=3,
            capthick=1.2,
            linewidth=1.2,
            markersize=5,
        )

    ax.axvline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.set_yticks(np.arange(len(sub)))
    ax.set_yticklabels(
        [f"{r['outcome']}  ({r['gender']})" for _, r in sub.iterrows()],
        fontsize=8,
    )
    ax.set_xlabel(f"{sc} of Diploma", fontsize=11)
    ax.set_title(f"Random Forest — {sc}", fontsize=12)
    ax.grid(axis="x", alpha=0.3, linestyle=":")

sig_patch = mpatches.Patch(color="#e74c3c", label="Significant (p ≤ 0.10)")
w_patch = mpatches.Patch(color="#3498db", label="Women — not significant")
m_patch = mpatches.Patch(color="#2ecc71", label="Men — not significant")
fig.legend(
    handles=[sig_patch, w_patch, m_patch],
    fontsize=9,
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, 1.01),
)

fig.suptitle(
    "Forest Plot: Effect of Higher Education on Health Outcomes\n"
    "(Double ML IRM, Random Forest, 90% CI)",
    fontsize=12,
    y=1.04,
)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (df_g, gender) in zip(axes, [(df_women, "Women"), (df_men, "Men")]):
    X = df_g[controls].values
    d = df_g[treatment].values.astype(int)
    clf = RandomForestClassifier(
        n_estimators=100, max_depth=5, min_samples_leaf=10, random_state=42
    )
    ps = cross_val_predict(clf, X, d, cv=5, method="predict_proba")[:, 1]

    sns.kdeplot(
        ps[d == 0],
        ax=ax,
        label="No diploma",
        fill=True,
        alpha=0.45,
        color="#3498db",
        bw_adjust=0.8,
    )
    sns.kdeplot(
        ps[d == 1],
        ax=ax,
        label="Has diploma",
        fill=True,
        alpha=0.45,
        color="#e74c3c",
        bw_adjust=0.8,
    )

    ax.set_xlim(0, 1)
    ax.set_xlabel("Propensity Score  P(diploma = 1 | X)", fontsize=11)
    ax.set_ylabel("Density", fontsize=11)
    ax.set_title(f"Propensity Score Overlap — {gender}", fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3, linestyle=":")

plt.suptitle(
    "Common Support Check: Propensity Score Distributions by Treatment Status",
    fontsize=13,
)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharey=True)

for ax, sc in zip(axes, ["ATE", "ATTE"]):
    sub = results_df[(results_df["model"] == "RF") & (results_df["score"] == sc)]

    for _, r in sub.iterrows():
        coef = r["coef"]
        neg_log_p = -np.log10(r["p_val"])
        marker = "o" if r["gender"] == "Women" else "s"

        if not r["significant"]:
            color, zorder, alpha_pt = "#aab0b5", 1, 0.6
        elif coef > 0:
            color, zorder, alpha_pt = "#e74c3c", 3, 0.9
        else:
            color, zorder, alpha_pt = "#3498db", 3, 0.9

        ax.scatter(
            coef,
            neg_log_p,
            color=color,
            marker=marker,
            s=65,
            zorder=zorder,
            alpha=alpha_pt,
            edgecolors="white",
            linewidths=0.4,
        )

        if r["significant"]:
            ax.annotate(
                f"{r['outcome']} ({r['gender'][0]})",
                (coef, neg_log_p),
                fontsize=7.5,
                xytext=(5, 3),
                textcoords="offset points",
            )

    ax.axhline(
        -np.log10(alpha),
        color="dimgray",
        linestyle="--",
        linewidth=1,
    )
    ax.axvline(0, color="dimgray", linestyle=":", linewidth=0.8, alpha=0.5)

    ax.set_xlabel(f"{sc} of Diploma", fontsize=11)
    ax.set_ylabel(r"$-\log_{10}$(p-value)", fontsize=11)
    ax.set_title(f"Random Forest — {sc}", fontsize=12)
    ax.grid(alpha=0.3, linestyle=":")

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="#e74c3c",
        markersize=8,
        label="Significant — positive (Women)",
    ),
    Line2D(
        [0],
        [0],
        marker="s",
        color="w",
        markerfacecolor="#e74c3c",
        markersize=8,
        label="Significant — positive (Men)",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="#3498db",
        markersize=8,
        label="Significant — negative (Women)",
    ),
    Line2D(
        [0],
        [0],
        marker="s",
        color="w",
        markerfacecolor="#3498db",
        markersize=8,
        label="Significant — negative (Men)",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="#aab0b5",
        markersize=8,
        label="Not significant (Women)",
    ),
    Line2D(
        [0],
        [0],
        marker="s",
        color="w",
        markerfacecolor="#aab0b5",
        markersize=8,
        label="Not significant (Men)",
    ),
    Line2D(
        [0],
        [0],
        linestyle="--",
        color="dimgray",
        label=f"Significance threshold (p = {alpha})",
    ),
]
fig.legend(
    handles=legend_handles,
    fontsize=8,
    loc="upper center",
    ncol=4,
    bbox_to_anchor=(0.5, 1.02),
)

fig.suptitle(
    "Volcano Plot: Effect of Higher Education on Health Outcomes\n"
    "(Double ML IRM, Random Forest)",
    fontsize=12,
    y=1.10,
)

plt.tight_layout()
plt.show()

### Выводы